# YOLOv8-pose fine-tune on Amateur Drawings Dataset

Fine-tunes YOLOv8n-pose on hand-drawn human figures.

**Downloads data directly from Meta** (annotations ~275 MB + tar ~50 GB).  
Extracts only the selected subset (3000 train + 500 val images).

**Before running:** `Runtime â†’ Change runtime type â†’ T4 GPU`

In [ ]:
import subprocess
r = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(r.stdout if r.returncode == 0 else 'No GPU found â€” switch runtime to GPU')

## 1. Install dependencies

In [ ]:
!pip install -q ultralytics

## 2. Download annotations from Meta (~275 MB)

In [ ]:
import json
from pathlib import Path

ANN_URL  = 'https://dl.fbaipublicfiles.com/amateur_drawings/amateur_drawings_annotations.json'
ANN_FILE = Path('/content/amateur_drawings_annotations.json')

TRAIN_COUNT = 3000  # @param {type:"integer"}
VAL_COUNT   = 500   # @param {type:"integer"}

if not ANN_FILE.exists():
    !wget -q --show-progress -O {ANN_FILE} {ANN_URL}
else:
    print(f'Annotations already present ({ANN_FILE.stat().st_size // 1024 // 1024} MB) — skipping.')

print('Loading annotations...')
with open(ANN_FILE) as f:
    ann = json.load(f)
print(f'Images: {len(ann["images"])}   Annotations: {len(ann["annotations"])}')

annotated_ids = {a['image_id'] for a in ann['annotations'] if a.get('num_keypoints', 0) > 0}
id_to_img = {img['id']: img for img in ann['images']}
# bare filename -> image_id (only annotated images)
fname_to_id = {
    Path(img['file_name']).name: img['id']
    for img in ann['images']
    if img['id'] in annotated_ids
}
print(f'Annotated images: {len(annotated_ids)}   Will extract first {TRAIN_COUNT + VAL_COUNT}')

## 3. Download & selectively extract tar (~50 GB download, extracts only 3500 images)

The full tar is ~50 GB but only the selected images are written to disk.  
Download takes ~20-40 min on Colab depending on network speed.

In [ ]:
import tarfile, time
from pathlib import Path

TAR_URL  = 'https://dl.fbaipublicfiles.com/amateur_drawings/amateur_drawings.tar'
TAR_FILE = Path('/content/amateur_drawings.tar')
BASE     = Path('/content/drawings')

for spl in ('train', 'val'):
    (BASE / 'images' / spl).mkdir(parents=True, exist_ok=True)
    (BASE / 'labels' / spl).mkdir(parents=True, exist_ok=True)

total_needed = TRAIN_COUNT + VAL_COUNT
already_done = sum(
    1 for spl in ('train', 'val')
    for _ in (BASE / 'images' / spl).glob('*')
)

train_ids, val_ids, id_to_split = [], [], {}

if already_done >= total_needed:
    print(f'Images already extracted ({already_done}) — rebuilding id maps from disk.')
    for spl, lst in [('train', train_ids), ('val', val_ids)]:
        for p in sorted((BASE / 'images' / spl).glob('*')):
            img_id = fname_to_id.get(p.name)
            if img_id is not None:
                lst.append(img_id)
                id_to_split[img_id] = spl
else:
    if not TAR_FILE.exists():
        print('Downloading tar (~50 GB, 20-40 min)...')
        !wget -q --show-progress -O {TAR_FILE} {TAR_URL}
        print('Download complete.')
    else:
        print(f'Tar already present ({TAR_FILE.stat().st_size // 1024 // 1024 // 1024} GB) — skipping download.')

    print(f'Extracting first {total_needed} annotated images from tar...')
    extracted = 0
    t0 = time.time()
    with tarfile.open(TAR_FILE, 'r') as tf:
        for member in tf:
            bare = Path(member.name).name
            img_id = fname_to_id.get(bare)
            if img_id is None:
                continue
            spl = 'train' if len(train_ids) < TRAIN_COUNT else 'val'
            dst = BASE / 'images' / spl / bare
            if not dst.exists():
                fobj = tf.extractfile(member)
                if fobj:
                    dst.write_bytes(fobj.read())
            (train_ids if spl == 'train' else val_ids).append(img_id)
            id_to_split[img_id] = spl
            extracted += 1
            if extracted % 100 == 0:
                print(f'  {extracted} / {total_needed}  ({time.time()-t0:.0f}s)', flush=True)
            if extracted >= total_needed:
                break
    print(f'Extracted {extracted} images in {time.time()-t0:.0f}s')

selected_ids = set(id_to_split)
for spl in ('train', 'val'):
    n = sum(1 for _ in (BASE / 'images' / spl).glob('*'))
    print(f'{spl}: {n} images')

## 4. Generate YOLO pose labels

In [ ]:
import cv2
from pathlib import Path

BASE = Path('/content/drawings')

def coco_to_yolo(a, w, h):
    bbox = a.get('bbox')
    kpts = a.get('keypoints')
    if not bbox or not kpts or a.get('num_keypoints', 0) == 0:
        return None
    bx, by, bw, bh = bbox
    cx = (bx + bw / 2) / w;  cy = (by + bh / 2) / h
    nw = bw / w;              nh = bh / h
    if not (0 < cx < 1 and 0 < cy < 1 and 0 < nw <= 1 and 0 < nh <= 1):
        return None
    parts = [f'{kpts[i]/w:.6f} {kpts[i+1]/h:.6f} {int(kpts[i+2])}' for i in range(0, len(kpts), 3)]
    return f'0 {cx:.6f} {cy:.6f} {nw:.6f} {nh:.6f} ' + ' '.join(parts)

img_to_anns = {}
for a in ann['annotations']:
    img_to_anns.setdefault(a['image_id'], []).append(a)

written = skipped = 0
for img_id in selected_ids:
    info  = id_to_img.get(img_id)
    split = id_to_split[img_id]
    bare  = Path(info['file_name']).name
    img_path = BASE / 'images' / split / bare
    if not img_path.exists():
        skipped += 1; continue
    img = cv2.imread(str(img_path))
    if img is None:
        skipped += 1; continue
    h, w = img.shape[:2]
    lines = [l for a in img_to_anns.get(img_id, []) if (l := coco_to_yolo(a, w, h))]
    if not lines:
        skipped += 1; continue
    (BASE / 'labels' / split / (img_path.stem + '.txt')).write_text('\n'.join(lines))
    written += 1

print(f'Labels written: {written}   skipped: {skipped}')

## 5. Write dataset.yaml

In [ ]:
from pathlib import Path

BASE = Path('/content/drawings')
yaml_text = f"""path: {BASE}
train: images/train
val:   images/val

nc: 1
names: ['drawn_humanoid']

kpt_shape: [17, 3]
flip_idx: [0, 2, 1, 4, 3, 6, 5, 8, 7, 10, 9, 12, 11, 14, 13, 16, 15]
"""
(BASE / 'dataset.yaml').write_text(yaml_text)
print((BASE / 'dataset.yaml').read_text())

## 6. Preview training data with keypoints

In [ ]:
import random, cv2, numpy as np, matplotlib.pyplot as plt
from pathlib import Path

BASE = Path('/content/drawings')
SKELETON = [
    (0,1),(0,2),(1,3),(2,4),
    (5,6),(5,7),(7,9),(6,8),(8,10),
    (5,11),(6,12),(11,12),
    (11,13),(13,15),(12,14),(14,16),
]

def draw_yolo_labels(img_path, label_path):
    img = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]
    for line in label_path.read_text().strip().splitlines():
        vals = list(map(float, line.split()))
        if len(vals) < 5 + 17 * 3:
            continue
        cx, cy, bw, bh = vals[1], vals[2], vals[3], vals[4]
        cv2.rectangle(img,
            (int((cx-bw/2)*w), int((cy-bh/2)*h)),
            (int((cx+bw/2)*w), int((cy+bh/2)*h)), (200,200,200), 1)
        pts = [(int(vals[5+k*3]*w), int(vals[5+k*3+1]*h), vals[5+k*3+2]) for k in range(17)]
        for j1, j2 in SKELETON:
            if pts[j1][2] > 0 and pts[j2][2] > 0:
                cv2.line(img, pts[j1][:2], pts[j2][:2], (80,200,80), 2)
        for px, py, v in pts:
            if v > 0:
                cv2.circle(img, (px, py), 5, (255,80,80), -1)
    return img

train_imgs = sorted((BASE / 'images/train').glob('*.png')) + sorted((BASE / 'images/train').glob('*.jpg'))
label_dir  = BASE / 'labels/train'
sample = random.sample(train_imgs, min(6, len(train_imgs)))

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
for ax, ip in zip(axes.flatten(), sample):
    lp = label_dir / (ip.stem + '.txt')
    if not lp.exists():
        ax.text(0.5, 0.5, 'label missing', ha='center'); ax.axis('off'); continue
    ax.imshow(draw_yolo_labels(ip, lp))
    ax.set_title(f'{ip.name}  ({len(lp.read_text().strip().splitlines())} ann)', fontsize=7)
    ax.axis('off')
for ax in axes.flatten()[len(sample):]:
    ax.axis('off')
plt.suptitle('YOLO pose labels â€” training set preview', fontsize=12)
plt.tight_layout()
plt.show()

## 7. Train

In [ ]:
import torch
from ultralytics import YOLO
from pathlib import Path

BASE      = Path('/content/drawings')
YAML_PATH = BASE / 'dataset.yaml'

EPOCHS = 50   # @param {type:"integer"}
BATCH  = 16   # @param {type:"integer"}
IMGSZ  = 640  # @param {type:"integer"}

device = 0 if torch.cuda.is_available() else 'cpu'
print(f'Device: {"GPU cuda:0" if device == 0 else "CPU"}')
if device == 'cpu':
    print('Warning: CPU training is very slow (~12-24h for 50 epochs). Consider --epochs 3 to verify the pipeline.')

model   = YOLO('yolov8n-pose.pt')
results = model.train(
    data=str(YAML_PATH),
    epochs=EPOCHS,
    batch=BATCH,
    imgsz=IMGSZ,
    device=device,
    workers=2,
    project='/content/runs',
    name='drawn_humanoid_pose',
    exist_ok=True,
    patience=15,
    save_period=10,
    plots=True,
)
print('Best model:', results.save_dir / 'weights/best.pt')

## 8. Export to ONNX (for Flutter / mobile)

In [ ]:
from ultralytics import YOLO
from pathlib import Path

best_pt = Path('/content/runs/drawn_humanoid_pose/weights/best.pt')
m = YOLO(str(best_pt))
m.export(format='onnx', imgsz=640, opset=12, simplify=True, dynamic=False)
onnx_path = best_pt.with_suffix('.onnx')
print(f'ONNX saved: {onnx_path}  ({onnx_path.stat().st_size // 1024 // 1024} MB)')

## 9. Download trained model

In [ ]:
import shutil
from google.colab import files

shutil.copy('/content/runs/drawn_humanoid_pose/weights/best.pt',   '/content/drawn_humanoid_pose.pt')
shutil.copy('/content/runs/drawn_humanoid_pose/weights/best.onnx', '/content/drawn_humanoid_pose.onnx')

files.download('/content/drawn_humanoid_pose.pt')
files.download('/content/drawn_humanoid_pose.onnx')
print('Downloaded: drawn_humanoid_pose.pt  +  drawn_humanoid_pose.onnx')

In [ ]:
# Optional: save to Google Drive so it survives session reset
from google.colab import drive
from pathlib import Path
import shutil

drive.mount('/content/drive', force_remount=False)
out_dir = Path('/content/drive/MyDrive/yolo_models')
out_dir.mkdir(parents=True, exist_ok=True)

shutil.copy('/content/drawn_humanoid_pose.pt',   out_dir / 'drawn_humanoid_pose.pt')
shutil.copy('/content/drawn_humanoid_pose.onnx', out_dir / 'drawn_humanoid_pose.onnx')
print(f'Saved to Drive: {out_dir}')

## 10. Test on validation images

In [ ]:
import random, cv2, numpy as np, matplotlib.pyplot as plt
from ultralytics import YOLO
from pathlib import Path

BASE    = Path('/content/drawings')
val_dir = BASE / 'images/val'

SKELETON_PAIRS = [
    (0,1),(0,2),(1,3),(2,4),
    (5,6),(5,7),(7,9),(6,8),(8,10),
    (5,11),(6,12),(11,12),
    (11,13),(13,15),(12,14),(14,16),
]

model  = YOLO('/content/runs/drawn_humanoid_pose/weights/best.pt')
imgs   = sorted(val_dir.glob('*.png')) + sorted(val_dir.glob('*.jpg'))
sample = random.sample(imgs, min(6, len(imgs)))

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
for ax, img_path in zip(axes.flatten(), sample):
    img_bgr = cv2.imread(str(img_path))
    results  = model(img_bgr, conf=0.01, verbose=False)
    vis      = img_bgr.copy()
    detected = False
    for r in results:
        if r.keypoints is None or len(r.keypoints.xy) == 0:
            continue
        kpts = r.keypoints.xy[0].cpu().numpy()
        if not np.any(kpts > 0):
            continue
        detected = True
        pts = [(int(kpts[k][0]), int(kpts[k][1])) for k in range(17)]
        for j1, j2 in SKELETON_PAIRS:
            if pts[j1] != (0,0) and pts[j2] != (0,0):
                cv2.line(vis, pts[j1], pts[j2], (80,200,80), 2)
        for px, py in pts:
            if (px, py) != (0,0):
                cv2.circle(vis, (px, py), 5, (0,80,255), -1)
        conf_val = float(r.boxes.conf[0]) if r.boxes is not None and len(r.boxes.conf) else 0
        ax.set_title(f'{img_path.name}\nconf={conf_val:.2f}', fontsize=7)
        break
    if not detected:
        ax.set_title(f'{img_path.name}\nno detection', fontsize=7)
    ax.imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB))
    ax.axis('off')

for ax in axes.flatten()[len(sample):]:
    ax.axis('off')
plt.suptitle('drawn_humanoid_pose.pt â€” validation set', fontsize=13)
plt.tight_layout()
plt.show()